# Knowledge Localization

**Survey Reference:** Section 12.2 - Knowledge Localization: Where Is Information Stored?

## Overview

**Knowledge localization** investigates where factual knowledge is stored in neural networks. Key techniques include:

- **Causal tracing:** Identify which components are necessary for knowledge recall
- **ROME/MEMIT:** Edit factual associations by modifying specific weights
- **Localization probes:** Find which layers encode specific facts

## Learning Objectives

1. Understand causal tracing methodology
2. Implement basic knowledge localization
3. Explore the role of MLP layers in storing facts
4. Understand the limitations of localization claims

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Causal Tracing

**Causal tracing** (Meng et al., 2022) identifies which activations are critical for factual recall:

1. Run model on prompt, get correct answer probability
2. Corrupt the subject tokens (e.g., add noise)
3. Selectively restore activations and measure recovery

In [ ]:
# Load model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

# Factual prompt
prompt = "The Eiffel Tower is located in"
answer = " Paris"  # Note: leading space for tokenization

In [ ]:
def get_answer_probability(model, tokenizer, prompt, answer):
    """Get probability of answer given prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    answer_id = tokenizer.encode(answer, add_special_tokens=False)[0]
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]  # Last position
        probs = F.softmax(logits, dim=-1)
    
    return probs[answer_id].item()

clean_prob = get_answer_probability(model, tokenizer, prompt, answer)
print(f"P('{answer}' | '{prompt}') = {clean_prob:.4f}")

In [ ]:
def corrupt_embeddings(embeddings, subject_positions, noise_level=3.0):
    """
    Corrupt embeddings at subject positions.
    """
    corrupted = embeddings.clone()
    for pos in subject_positions:
        noise = torch.randn_like(corrupted[0, pos, :]) * noise_level
        corrupted[0, pos, :] += noise
    return corrupted

def find_subject_positions(tokenizer, prompt, subject):
    """Find token positions of subject in prompt."""
    tokens = tokenizer.tokenize(prompt)
    subject_tokens = tokenizer.tokenize(subject)
    
    positions = []
    for i in range(len(tokens)):
        if tokens[i:i+len(subject_tokens)] == subject_tokens:
            positions.extend(range(i, i + len(subject_tokens)))
    
    return positions

subject = "Eiffel Tower"
subject_positions = find_subject_positions(tokenizer, prompt, subject)
print(f"Subject positions: {subject_positions}")

## 2. Implementing Causal Tracing

In [ ]:
class CausalTracingHook:
    """
    Hook for causal tracing: restore clean activations at specific positions.
    """
    
    def __init__(self, clean_activations, restore_positions):
        self.clean_activations = clean_activations
        self.restore_positions = restore_positions
    
    def __call__(self, module, input, output):
        hidden_states = output[0]
        for pos in self.restore_positions:
            hidden_states[0, pos, :] = self.clean_activations[0, pos, :]
        return (hidden_states,) + output[1:]


def causal_trace(
    model,
    tokenizer,
    prompt: str,
    answer: str,
    subject_positions: list[int],
    noise_level: float = 3.0,
):
    """
    Perform causal tracing to find which layers/positions matter.
    
    Returns:
        results: [n_layers, n_positions] matrix of recovery effects
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    answer_id = tokenizer.encode(answer, add_special_tokens=False)[0]
    n_positions = inputs.input_ids.shape[1]
    n_layers = model.config.n_layer
    
    # Get clean activations
    with torch.no_grad():
        clean_outputs = model(**inputs, output_hidden_states=True)
    clean_hidden_states = clean_outputs.hidden_states
    clean_prob = F.softmax(clean_outputs.logits[0, -1], dim=-1)[answer_id].item()
    
    # Get corrupted baseline
    # (This is simplified - full implementation needs embedding corruption)
    
    # For each layer and position, restore and measure
    results = np.zeros((n_layers, n_positions))
    
    # TODO: Implement full causal tracing
    # For each (layer, position), corrupt subject and restore that specific activation
    
    return results

## 3. The Role of MLP Layers

Research suggests factual knowledge is primarily stored in MLP layers:
- **MLP** ≈ key-value memory
- **Attention** routes information to MLPs
- **Mid-to-late layers** contain most factual associations

In [ ]:
def analyze_mlp_contribution(
    model,
    tokenizer,
    prompt: str,
    answer: str,
):
    """
    Measure each MLP layer's contribution to the answer.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    answer_id = tokenizer.encode(answer, add_special_tokens=False)[0]
    
    # Get full model probability
    with torch.no_grad():
        full_output = model(**inputs)
    full_prob = F.softmax(full_output.logits[0, -1], dim=-1)[answer_id].item()
    
    # Ablate each MLP and measure effect
    mlp_effects = []
    n_layers = model.config.n_layer
    
    for layer_idx in range(n_layers):
        # Hook to zero out MLP output
        def ablate_mlp(module, input, output):
            return torch.zeros_like(output)
        
        mlp = model.transformer.h[layer_idx].mlp
        handle = mlp.register_forward_hook(ablate_mlp)
        
        with torch.no_grad():
            ablated_output = model(**inputs)
        ablated_prob = F.softmax(ablated_output.logits[0, -1], dim=-1)[answer_id].item()
        
        handle.remove()
        
        effect = full_prob - ablated_prob
        mlp_effects.append(effect)
    
    return mlp_effects

effects = analyze_mlp_contribution(model, tokenizer, prompt, answer)

plt.figure(figsize=(10, 5))
plt.bar(range(len(effects)), effects)
plt.xlabel('Layer')
plt.ylabel('Effect on P(answer)')
plt.title('MLP Layer Contribution to Factual Recall')
plt.show()

## 4. Knowledge Editing (ROME)

**ROME** (Rank-One Model Editing) modifies specific facts by updating MLP weights.

In [ ]:
def rome_style_edit(
    model,
    layer_idx: int,
    key_vector: torch.Tensor,  # What triggers the fact
    value_vector: torch.Tensor,  # What should be output
    scale: float = 1.0,
):
    """
    Simplified ROME-style weight edit.
    
    Adds a rank-one update to MLP weights: W' = W + value @ key^T
    """
    # Get MLP weights
    mlp = model.transformer.h[layer_idx].mlp
    
    # The actual ROME implementation is more complex, involving:
    # 1. Computing the key (subject representation)
    # 2. Solving for the value that produces the new fact
    # 3. Computing a low-rank update that minimizes side effects
    
    # This is a placeholder for the concept
    pass

# Full ROME implementation requires significant additional code
# See: https://github.com/kmeng01/rome

## 5. Limitations of Localization

Important caveats:

1. **Distributed representations:** Knowledge may be spread across components
2. **Redundancy:** Same fact may be stored in multiple places
3. **Context dependence:** Localization may depend on how you probe
4. **Causal ≠ storage:** A component being important doesn't mean it "stores" the fact

In [ ]:
# TODO: Demonstrate distributed nature of knowledge

## Key Takeaways

1. **Causal tracing** identifies which components matter for factual recall
2. **MLP layers** appear to store factual associations
3. **Mid-to-late layers** are most important for facts
4. **Knowledge editing** (ROME/MEMIT) can modify specific facts
5. **Localization has limits:** Knowledge is often distributed

## Next Steps

→ **03_representation_engineering.ipynb**: Steer model behavior by manipulating representations